## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 04: Primary Difference-in-Differences Analysis**

---

**Scope:** Estimates the headline result: did Bank Nifty's expiry-day trading concentration rise more than Nifty 50's after the weekly-options removal?

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, expiry calendar, holidays, F&O Bhavcopy archive
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical/volatility features; lot-size-invariant turnover and open interest
3. **03_weekly_options_discontinuation_identification.ipynb** -- dating the Nov 2024 weekly-contract removal directly from exchange records
4. **04_primary_diff_in_diff_analysis.ipynb** -- the headline difference-in-differences result (Bank Nifty vs. Nifty 50)
5. **05_robustness_checks.ipynb** -- placebo cutover dates, leave-one-out, event-study leads/lags
6. **06_expiry_volume_event_study.ipynb** -- descriptive expiry-day volume/OI patterns by regime
7. **07_walk_forward_predictability.ipynb** -- secondary analysis: leakage-free return-predictability test (null result)
8. **08_interrupted_time_series.ipynb** -- secondary analysis: settlement-weekday effect on returns/volatility (null result)
9. **09_figures_for_paper.ipynb** -- regenerates all manuscript figures

**Current location:** **04_primary_diff_in_diff_analysis.ipynb** <- *you are here*


`scripts.diff_in_diff_banknifty_vs_nifty` pools daily Bank Nifty and Nifty 50 turnover (and, separately, open interest) and estimates

```
log(Value_it) = a + b1*Expiry_it + b2*MonthlyOnly_t + b3*BankNifty_i
    + b4*(Expiry_it x MonthlyOnly_t) + b5*(Expiry_it x BankNifty_i)
    + b6*(MonthlyOnly_t x BankNifty_i) + b7*(Expiry_it x MonthlyOnly_t x BankNifty_i)
    + b8*Trend_t + weekday fixed effects + e_it
```

with Newey-West standard errors. The coefficient on the triple interaction (b7) is the causal estimate of interest.

In [1]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

Project root: E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml


In [2]:
from scripts.diff_in_diff_banknifty_vs_nifty import load_panel, triple_interaction_regression

panel = load_panel()
model_value = triple_interaction_regression(panel, 'Total_Value')
model_oi = triple_interaction_regression(panel, 'Total_OpenInterest_Notional')
print('=== log(Total_Value) ===')
print(model_value.summary().tables[1])

=== log(Total_Value) ===
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                           29.2106      0.048    606.513      0.000      29.116      29.305
IsExpiry                         0.2182      0.063      3.447      0.001       0.094       0.342
IsMonthlyOnly                   -0.4395      0.070     -6.244      0.000      -0.577      -0.302
IsBankNifty                      0.2531      0.091      2.783      0.005       0.075       0.431
Expiry_x_Monthly                 0.4114      0.136      3.035      0.002       0.146       0.677
Expiry_x_BankNifty               0.4340      0.078      5.557      0.000       0.281       0.587
Monthly_x_BankNifty             -3.2220      0.158    -20.396      0.000      -3.532      -2.912
Expiry_x_Monthly_x_BankNifty     1.3418      0.262      5.113      0.000       0.827       1.856
Trend

In [3]:
print('=== log(Total_OpenInterest_Notional) ===')
print(model_oi.summary().tables[1])

=== log(Total_OpenInterest_Notional) ===
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                           22.8480      0.029    790.168      0.000      22.791      22.905
IsExpiry                         0.0751      0.030      2.469      0.014       0.015       0.135
IsMonthlyOnly                    0.7291      0.071     10.290      0.000       0.590       0.868
IsBankNifty                     -2.8782      0.073    -39.635      0.000      -3.021      -2.736
Expiry_x_Monthly                -0.0396      0.071     -0.558      0.577      -0.179       0.099
Expiry_x_BankNifty               0.1386      0.048      2.876      0.004       0.044       0.233
Monthly_x_BankNifty             -1.7738      0.115    -15.412      0.000      -1.999      -1.548
Expiry_x_Monthly_x_BankNifty     0.2497      0.100      2.486      0.013       0.053  

### Reading the result

The `Expiry_x_Monthly_x_BankNifty` coefficient is the difference-in-differences estimate: how much more Bank Nifty's expiry-day activity rose, relative to Nifty 50, after the weekly contract was discontinued. A positive, significant coefficient on both turnover and open interest is the paper's primary finding.